## Why Evaluation Matter

In [1]:
# Scenario: Walmart Pickup Assistant — Evaluating AI responses for accuracy, reliability, and customer usefulness.

# Imagine Walmart has developed an AI-powered customer support assistant that helps customers with their online pickup orders.

# Customers ask questions such as:
#     - Is my order ready for pickup?
#     - Where should I park my vehicle?
#     - How long will my order take?
#     - What should I do if my order cannot be found?
# The AI assistant looks up order information and generates a response.

# But here is the challenge: What happens when the AI gives an incorrect answer, even though it sounds confident and professional?

# Customer's question
# Order WM-104
#     "Is my order ready? Where should I park?"

# ACTUAL ORDER INFORMATION
#     Order ready. Park in Bay 3.

# Correct AI
#     "Your order is ready. Park in Bay 3."

# Incorrect AI
#     "Your order is ready. Park in Bay 5."
# The customer might park in the wrong location and experience delays.

# This is the central problem addressed by the notebook: How do we systematically check whether an AI assistant is producing correct, helpful, and safe responses before trusting it in a real business environment?

In [2]:
# The notebook divides this problem into four activities.

# Step 1: Identify bad AI responses
# Step 2: Define evaluation criteria
# Step 3: Compare human and automated evaluation
# Step 4: Measure whether AI quality deteriorates

In [3]:
# Evaluation helps the business answer three important questions:
# 1. Is the AI providing information supported by the available evidence?
# 2. Is the response actually helping the customer take the correct next action?
# 3. Has AI performance improved or deteriorated over time?

## 1. Analyze poor-quality outputs 

Review the same four requests at two time snapshots: **Week 1** and **Week 4**. Before looking at any labels, compare each reply with the lookup evidence. In pairs, identify:

1. A fabricated or unsupported operational detail (a hallucination).
2. A silent failure: a reply is delivered but leaves the customer without a needed next step.
3. A safe paraphrase that a rigid word-match might miss.

In [ ]:
# Problem statement: Walmart has an AI assistant that answers customer questions about pickup orders. We want to compare its responses in Week 1 and Week 4 to identify whether the AI has started making mistakes.

In [4]:
# Import pandas when we build the matched classroom panel.
import pandas as pd

# Keep requests and evidence fixed while comparing two points in time.
cases = pd.DataFrame([
    ("ready", "ready", "Week 1", "Is WM-104 ready? Where do I park?",
     "Ready; bay 3", "Your order is ready. Park in bay 3."),
    ("ready", "ready", "Week 4", "Is WM-104 ready? Where do I park?",
     "Ready; bay 3", "Your order is ready. Park in bay 5."),
    ("preparing", "preparing", "Week 1", "Can I collect WM-105 now?",
     "Still preparing; no pickup time available",
     "Your order is still being prepared. We do not have a pickup time yet."),
    ("preparing", "preparing", "Week 4", "Can I collect WM-105 now?",
     "Still preparing; no pickup time available",
     "Your order should be ready in 10 minutes."),
    ("timeout", "lookup unavailable", "Week 1", "Can I collect WM-106 now?",
     "Lookup timed out; status and bay unknown",
     "I could not verify your status. Please ask a pickup associate."),
    ("timeout", "lookup unavailable", "Week 4", "Can I collect WM-106 now?",
     "Lookup timed out; status and bay unknown",
     "Thanks for checking. We will see you soon."),
    ("missing", "lookup unavailable", "Week 1", "Where do I park for WM-107?",
     "No record found; status and bay unknown",
     "I could not verify a bay. Please ask a pickup associate."),
    ("missing", "lookup unavailable", "Week 4", "Where do I park for WM-107?",
     "No record found; status and bay unknown",
     "I could not confirm your bay. A curbside team member can help."),
], columns=["case", "slice", "snapshot", "request", "evidence", "response"])

# Show a blank review worksheet; label each response before moving on.
review_worksheet = cases[["case", "snapshot", "request", "evidence", "response"]].copy()
review_worksheet["grounded?"] = ""
review_worksheet["actionable?"] = ""
review_worksheet["critical failure?"] = ""
with pd.option_context("display.max_colwidth", 80):
    display(review_worksheet)

,case,snapshot,request,evidence,response,grounded?,actionable?,critical failure?
0,ready,Week 1,Is WM-104 ready? Where do I park?,Ready; bay 3,Your order is ready. Park in bay 3.,,,
1,ready,Week 4,Is WM-104 ready? Where do I park?,Ready; bay 3,Your order is ready. Park in bay 5.,,,
2,preparing,Week 1,Can I collect WM-105 now?,Still preparing; no pickup time available,Your order is still being prepared. We do not have a pickup time yet.,,,
3,preparing,Week 4,Can I collect WM-105 now?,Still preparing; no pickup time available,Your order should be ready in 10 minutes.,,,
4,timeout,Week 1,Can I collect WM-106 now?,Lookup timed out; status and bay unknown,I could not verify your status. Please ask a pickup associate.,,,
5,timeout,Week 4,Can I collect WM-106 now?,Lookup timed out; status and bay unknown,Thanks for checking. We will see you soon.,,,
6,missing,Week 1,Where do I park for WM-107?,No record found; status and bay unknown,I could not verify a bay. Please ask a pickup associate.,,,
7,missing,Week 4,Where do I park for WM-107?,No record found; status and bay unknown,I could not confirm your bay. A curbside team member can help.,,,


## 2. Define evaluation criteria 

Before running the answer-key cell, agree on criteria that follow from the evidence and customer task:

- **Grounded**: operational claims agree with the lookup result; no bay, status, or ETA is invented.
- **Actionable**: the reply answers the request, or gives a useful next step when the lookup cannot verify it.
- **Critical failure**: a wrong pickup instruction, invented ETA, or missing handoff when the status is unknown.

For each row, write `True` or `False` for each criterion and one short reason. Then compare your judgments with the sample reviewer key. The key is an example rubric application, not universal ground truth.

In [5]:
# Store one auditable example review for every matched case and snapshot.
annotations = pd.DataFrame([
    ("ready", "Week 1", True, True, False, "Verified status and bay"),
    ("ready", "Week 4", False, False, True, "Bay 5 contradicts verified bay 3"),
    ("preparing", "Week 1", True, True, False, "No unsupported pickup time"),
    ("preparing", "Week 4", False, False, True, "Invented 10-minute ETA"),
    ("timeout", "Week 1", True, True, False, "Discloses uncertainty and gives a handoff"),
    ("timeout", "Week 4", False, False, True, "Implies collection is imminent and omits handoff"),
    ("missing", "Week 1", True, True, False, "Discloses missing evidence and gives a handoff"),
    ("missing", "Week 4", True, True, False, "Safe paraphrase with an actionable handoff"),
], columns=["case", "snapshot", "grounded", "actionable", "critical", "reason"])

# Join labels to the matching output and validate the one-to-one pairing.
review = cases.merge(annotations, on=["case", "snapshot"], validate="one_to_one")
display(review[["case", "snapshot", "grounded", "actionable", "critical", "reason"]])

,case,snapshot,grounded,actionable,critical,reason
0,ready,Week 1,True,True,False,Verified status and bay
1,ready,Week 4,False,False,True,Bay 5 contradicts verified bay 3
2,preparing,Week 1,True,True,False,No unsupported pickup time
3,preparing,Week 4,False,False,True,Invented 10-minute ETA
4,timeout,Week 1,True,True,False,Discloses uncertainty and gives a handoff
5,timeout,Week 4,False,False,True,Implies collection is imminent and omits handoff
6,missing,Week 1,True,True,False,Discloses missing evidence and gives a handoff
7,missing,Week 4,True,True,False,Safe paraphrase with an actionable handoff


## 3. Compare qualitative review with a simple heuristic 

Qualitative review uses context and the agreed rubric; it can recognize a safe paraphrase, but reviewers need consistent criteria. A heuristic is cheap and repeatable, but exact phrase rules can reject valid wording or miss subtle errors.

The next cell applies a deliberately small set of checks to the same panel. Inspect every disagreement with the sample reviewer labels. Discuss which checks are useful as alerts and why they should not be treated as an automatic release decision.

In [6]:
# Use only standard-library text checks to compare a simple heuristic with human review.
import re

def heuristic_review(row):
    # Normalize the reply before checking for case-specific evidence.
    answer = row.response.lower()


    # Scenario 1: Order is ready
    if row.case == "ready":
        # Require the verified bay and reject any conflicting bay number.
        expected_bay = re.search(r"\bbay\s+(\d+)\b", row.evidence.lower()).group(1)
        observed_bays = set(re.findall(r"\bbay\s+(\d+)\b", answer))
        grounded = observed_bays == {expected_bay}
        actionable = grounded and "ready" in answer

    # Scenario 2: Order is still preparing
    elif row.case == "preparing":
        # A preparing status is useful only when the reply does not invent an ETA.
        invented_eta = bool(re.search(r"\b\d+\s+minutes?\b", answer)) # "Ready in 10 minutes" → True, - "Still being prepared" → False
        grounded = "prepar" in answer and not invented_eta
        actionable = grounded

    # Scenario 3 and 4: Timeout or missing order
    else:
        # Intentionally use narrow phrases to expose the limits of exact-word checks.
        uncertainty = "could not verify" in answer or "could not confirm" in answer
        grounded = uncertainty
        actionable = uncertainty and "associate" in answer

    return pd.Series({"heuristic_grounded": grounded, "heuristic_actionable": actionable})

# Compare automated flags with the reviewer labels case by case.
comparison = review.join(review.apply(heuristic_review, axis=1))
comparison["grounded_disagreement"] = comparison["grounded"] != comparison["heuristic_grounded"]
comparison["actionable_disagreement"] = comparison["actionable"] != comparison["heuristic_actionable"]
display(comparison[[
    "case", "snapshot", "grounded", "heuristic_grounded",
    "actionable", "heuristic_actionable",
    "grounded_disagreement", "actionable_disagreement",
]])

# Reopen mismatches instead of hiding them in an average.
display(comparison.loc[
    comparison["grounded_disagreement"] | comparison["actionable_disagreement"],
    ["case", "snapshot", "evidence", "response", "reason"],
])

,case,snapshot,grounded,heuristic_grounded,actionable,heuristic_actionable,grounded_disagreement,actionable_disagreement
0,ready,Week 1,True,True,True,True,False,False
1,ready,Week 4,False,False,False,False,False,False
2,preparing,Week 1,True,True,True,True,False,False
3,preparing,Week 4,False,False,False,False,False,False
4,timeout,Week 1,True,True,True,True,False,False
5,timeout,Week 4,False,False,False,False,False,False
6,missing,Week 1,True,True,True,True,False,False
7,missing,Week 4,True,True,True,False,False,True


,case,snapshot,evidence,response,reason
7,missing,Week 4,No record found; status and bay unknown,I could not confirm your bay. A curbside team ...,Safe paraphrase with an actionable handoff


## 4. Measure output drift over time 

Compare **Week 1** with **Week 4** on the same four requests. Use only a few practical measures:

- **Grounded rate**: share of reviewed replies whose operational claims match evidence.
- **Actionable rate**: share that answers the request or gives the required handoff.
- **Critical-failure count**: count of release-blocking errors; never hide this in an average.
- **Slice-level actionable rate**: shows which workflow category regressed.

Apply the sample gate of zero critical failures, then inspect the specific Week 4 regressions. Four examples per snapshot are intentionally small for teaching: a real release decision needs a larger reviewed panel and adequate coverage of critical cases.

In [7]:
# Report interpretable rates and critical counts for each time snapshot.
metrics = review.groupby("snapshot", sort=False).agg(
    examples=("case", "size"),
    grounded_rate=("grounded", "mean"),
    actionable_rate=("actionable", "mean"),
    critical_failures=("critical", "sum"),
)
metrics["sample_gate"] = metrics["critical_failures"].eq(0).map({True: "PASS", False: "BLOCK"})
display(metrics)

# Break out actionability by workflow slice so an aggregate cannot hide a regression.
by_slice = review.pivot_table(
    index="slice", columns="snapshot", values="actionable", aggfunc="mean"
)
by_slice["change_pp"] = 100 * (by_slice["Week 4"] - by_slice["Week 1"])
display(by_slice)

# Inspect the reviewed Week 4 critical cases behind the sample gate.
display(review.loc[
    (review["snapshot"] == "Week 4") & review["critical"],
    ["case", "evidence", "response", "reason"],
])

,examples,grounded_rate,actionable_rate,critical_failures,sample_gate
snapshot,,,,,
Week 1,4,1.00,1.00,0,PASS
Week 4,4,0.25,0.25,3,BLOCK


snapshot,Week 1,Week 4,change_pp
slice,,,
lookup unavailable,1.0,0.5,-50.0
preparing,1.0,0.0,-100.0
ready,1.0,0.0,-100.0


,case,evidence,response,reason
1,ready,Ready; bay 3,Your order is ready. Park in bay 5.,Bay 5 contradicts verified bay 3
3,preparing,Still preparing; no pickup time available,Your order should be ready in 10 minutes.,Invented 10-minute ETA
5,timeout,Lookup timed out; status and bay unknown,Thanks for checking. We will see you soon.,Implies collection is imminent and omits handoff


**Takeaway** · On this matched toy panel, Week 4 regresses: the wrong bay, invented ETA, and missing handoff produce three critical failures. The safe paraphrase also shows why heuristic mismatches need human review. The rates describe this small panel only; they are not estimates of production quality.

Reference · [pandas GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)